In [2]:
!pip install -r requirements.txt

In [3]:
import os
import asyncio
import operator
from typing import Annotated, Sequence, TypedDict
import json
from dotenv import load_dotenv

from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_community.document_transformers import Html2TextTransformer
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.messages import BaseMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool

from langgraph.graph import StateGraph, END
from langgraph.prebuilt import tools_condition
import getpass

USER_AGENT environment variable not set, consider setting it to identify your requests.
/opt/miniconda3/envs/langchain-test-env/lib/python3.13/site-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [4]:
import warnings
warnings.filterwarnings("ignore")
os.environ["OPENAI_API_KEY"] = "sk-d02e0aefe1be44e59b51f336a734ce20"
os.environ["USER_AGENT"] = "travel_agent/1.0"

print("key loaded:", bool(os.environ.get("OPENAI_API_KEY")))

key loaded: True


In [5]:
UK_DESTINATIONS = [ #A
    "Cornwall",
    "North_Cornwall",
    "South_Cornwall",
    "West_Cornwall",
]

In [6]:
async def save_into_text(destinations: Sequence[str]):
    """Download WikiVoyage pages and save into local"""
    urls = [f"https://en.wikivoyage.org/wiki/{slug}" 
        for slug in destinations] #C
    loader = AsyncHtmlLoader(urls) #C
    print("Downloading destination pages ...") #C
    docs = await loader.aload() #C
    html2text = Html2TextTransformer()
    docs_transformed = html2text.transform_documents(docs)
    with open("travel.txt", "w", encoding="utf-8") as f:
        for doc in docs_transformed:
            f.write(doc.page_content)
            f.write("\n\n")

In [7]:
await save_into_text(UK_DESTINATIONS)

Fetching pages: 100%|##########| 4/4 [00:01<00:00,  2.15it/s]


In [8]:
# ý tưởng: Viết 1 tool đọc text sau đó trả về toàn bộ văn bản trong file .txt
# ý tưởng: Sau đó lấy đoạn text + query gửi lên LLM để 
import difflib # Cần import thư viện này để dùng SequenceMatcher

@tool(description="Search travel information about destinations in England.")
def search_travel_info(query: str) -> str:
    """ Search data from file travel.txt based on fuzzy search"""
    try:
        with open("travel.txt", "r", encoding="utf-8") as f:
            content = f.read()
            
        # Tách nội dung thành các đoạn văn (paragraphs), bỏ qua các đoạn quá ngắn
        paragraphs = [p.strip() for p in content.split("\n\n") if len(p.strip()) > 30]
        
        # Hàm tính độ tương đồng giữa truy vấn (query) và một đoạn văn bản
        def get_similarity(paragraph):
            # So sánh không phân biệt chữ hoa/thường để tăng độ chính xác
            return difflib.SequenceMatcher(None, query.lower(), paragraph.lower()).ratio()
            
        # Sắp xếp các đoạn văn theo độ tương đồng giảm dần (cao nhất xếp trước)
        sorted_paragraphs = sorted(paragraphs, key=get_similarity, reverse=True)
        
        # Lấy top 3 đoạn văn có nội dung giống với câu truy vấn nhất
        top_results = sorted_paragraphs[:10]
        if not top_results:
            return "No matching information found."
        
        # Ghép các kết quả lại và ngăn cách bằng '---'
        return "\n---\n".join(top_results)
        
    except FileNotFoundError:
        return "Error: File 'travel.txt' not found."
    except Exception as e:
        return f"Error: {e}"

@tool(description="Get the current weather information and temperature for destinations in England (e.g. Cornwall, North_Cornwall, etc.)")
def get_weather_info(city: str) -> str:
    """Returns the weather condition and temperature for a given city."""
    
    # Chuẩn hóa đầu vào để dễ bề so sánh (viết thường, đổi khoảng trắng thành _)
    city_normalized = city.strip().lower().replace(" ", "_")
    
    # Dữ liệu thời tiết giả lập (Mock Data) cho các vùng của Cornwall
    mock_weather_db = {
        "cornwall": "Sunny, 18°C, perfect for a beach day.",
        "north_cornwall": "Windy and partly cloudy, 15°C. Great for surfing.",
        "south_cornwall": "Clear skies, 19°C. Ideal for sailing.",
        "west_cornwall": "Light rain, 14°C. Good for indoor activities or coastal walks."
    }
    
    # Nếu tìm thấy thành phố trong Database giả lập
    if city_normalized in mock_weather_db:
        return mock_weather_db[city_normalized]
    
    # Nếu hỏi các thành phố khác (ví dụ: London, Manchester), ta random một thời tiết bất kỳ
    conditions = ["Sunny", "Cloudy", "Rainy", "Windy", "Foggy"]
    temp = random.randint(10, 25)
    random_condition = random.choice(conditions)
    
    return f"The current weather in {city} is {random_condition} with a temperature of {temp}°C."

In [9]:
TOOLS = [search_travel_info, get_weather_info] #A
llm_model = ChatOpenAI(
    model="Qwen3.6-27B", 
    base_url="https://ai-fit.hcmus.edu.vn/api/v1",  # <-- Chú ý sửa chính xác thành /api/v1
    api_key=os.environ.get("OPENAI_API_KEY")
)
llm_with_tools = llm_model.bind_tools(TOOLS) #C

In [10]:
class AgentState(TypedDict): #A
    messages: Annotated[Sequence[BaseMessage], operator.add] #B

In [11]:
class ToolsExecutionNode: #A
    """Execute tools requested by the LLM in the last AIMessage."""

    def __init__(self, tools: Sequence): #B
        self._tools_by_name = {t.name: t for t in tools}

    def __call__(self, state: dict): #C
        messages: Sequence[BaseMessage] = state.get("messages", [])  

        last_msg = messages[-1] #D
        tool_messages: list[ToolMessage] = [] #E
        tool_calls = getattr(last_msg, 
            "tool_calls", []) #F
        
        for tool_call in tool_calls: #G
            tool_name = tool_call["name"] #H
            tool_args = tool_call["args"] #I
            tool = self._tools_by_name[tool_name] #J
            result = tool.invoke(tool_args) #K
            tool_messages.append(
                ToolMessage(
                    content=json.dumps(result), #L
                    name=tool_name,
                    tool_call_id=tool_call["id"],
                )
            )
        return {"messages": tool_messages} #M

In [12]:
tools_execution_node = ToolsExecutionNode(TOOLS) #N

In [13]:
def llm_node(state: AgentState): #A    
    """LLM node that decides whether 
    to call the search tool."""
    current_messages = state["messages"] #B
    respose_message = llm_with_tools.invoke(
        current_messages) #C

    return {"messages": [respose_message]} #D

In [14]:
builder = StateGraph(AgentState) #A
builder.add_node("llm_node", llm_node) #B
builder.add_node("tools", tools_execution_node) #B

builder.add_conditional_edges("llm_node", 
    tools_condition) #C

builder.add_edge("tools", "llm_node") #D

builder.set_entry_point("llm_node") #E
travel_info_agent = builder.compile() #F

In [15]:
def chat_loop():
    print("UK Travel Assistant (type 'exit' to quit)")
    
    # Thêm SystemMessage để định hướng LLM bắt buộc phải dùng tool
    from langchain_core.messages import SystemMessage
    system_msg = SystemMessage(content="You are a UK travel assistant. You MUST use the search_travel_info tool to look up information before answering the user's question. Do not answer from your own knowledge.")
    
    while True:
        user_input = input("You: ").strip()
        if user_input.lower() in {"exit", "quit"}:
            break
            
        # Truyền cả SystemMessage và HumanMessage vào state
        state = {"messages": [system_msg, HumanMessage(content=user_input)]}
        
        result = travel_info_agent.invoke(state)
        response_msg = result["messages"][-1]
        print(f"Assistant: {response_msg.content}\n")

In [16]:
chat_loop()
#Cornwall

UK Travel Assistant (type 'exit' to quit)
Assistant: It seems that the specific information about London in the search results was limited, but I can tell you that it took approximately 3 hours and 20 minutes by train from London to Plymouth. If you need more detailed information about London, please let me know!

